# Q1: Corpus-Based Chatbot

Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.preprocessing import MinMaxScaler
import os

# 1. Device Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# 2. Robust Data Loading
def safe_load_csv(file_path):
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found.")
        return None
    return pd.read_csv(file_path, engine='python', on_bad_lines='skip', quoting=0, escapechar='\\')

train_df = safe_load_csv('trac2_CONVT_train.csv')
dev_df = safe_load_csv('trac2_CONVT_dev.csv')
test_df = safe_load_csv('project_part2_test.csv')

# 3. Pre-processing & Embedding
scaler = MinMaxScaler()
train_df[['Emotion_norm', 'Empathy_norm']] = scaler.fit_transform(train_df[['Emotion', 'Empathy']])

# Initialize the model
st_model = SentenceTransformer('all-MiniLM-L6-v2').to(DEVICE)

# Updated Encoding block with progress bar and batching
print("Encoding training corpus (Starting)...")
train_embeddings = st_model.encode(
    train_df['text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,  # This shows the blue progress bar
    batch_size=64           # Processes 64 sentences at a time for speed
)
print("Encoding Complete!")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding training corpus (Starting)...


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Encoding Complete!


The Core Logic (Similarity Function)

In [2]:
def get_corpus_response(history_text, target_emo, target_emp, target_pol, train_df, train_embeddings):
    # Tunable weights as per Eq 5 in instructions
    w1, w2, w3, w4 = 0.4, 0.2, 0.2, 0.2

    query_emb = st_model.encode(history_text, convert_to_tensor=True)
    stext = util.cos_sim(query_emb, train_embeddings).squeeze()

    target_emo_norm = target_emo / 5.0
    semotion = 1 - torch.abs(target_emo_norm - torch.tensor(train_df['Emotion_norm'].values, device=DEVICE))

    target_emp_norm = target_emp / 5.0
    sempathy = 1 - torch.abs(target_emp_norm - torch.tensor(train_df['Empathy_norm'].values, device=DEVICE))

    spolarity = (torch.tensor(train_df['EmotionalPolarity'].values, device=DEVICE) == target_pol).float()

    stotal = (w1 * stext) + (w2 * semotion) + (w3 * sempathy) + (w4 * spolarity)
    return train_df.iloc[torch.argmax(stotal).item()]['text']

Test Set Generation

In [3]:
print("Generating responses for project_part2_test.csv...")
test_results = []
unique_test_ids = test_df['conversation_id'].unique()

for conv_id in unique_test_ids:
    conv_data = test_df[test_df['conversation_id'] == conv_id].sort_values('turn_id')
    for current_turn in range(6, 11):
        target_row = conv_data[conv_data['turn_id'] == current_turn]
        if not target_row.empty:
            history_text = " [SEP] ".join(conv_data[conv_data['turn_id'] < current_turn]['text'].tolist())
            gen_text = get_corpus_response(history_text, target_row.iloc[0]['Emotion'],
                                           target_row.iloc[0]['Empathy'],
                                           target_row.iloc[0]['EmotionalPolarity'],
                                           train_df, train_embeddings)
            test_results.append({'id': target_row.iloc[0]['id'], 'turn_number': current_turn, 'generated_response': gen_text})

pd.DataFrame(test_results).to_csv('generations_corpus.csv', index=False)
print("Saved generations_corpus.csv")

Generating responses for project_part2_test.csv...
Saved generations_corpus.csv


Dev Set Evaluation

In [5]:
!pip install evaluate rouge_score bert_score
import evaluate

print("Evaluating on Validation (Dev) Set for metrics...")
rouge = evaluate.load('rouge'); bleu = evaluate.load('bleu'); bertscore = evaluate.load('bertscore')

dev_predictions = []
dev_references = []
unique_dev_ids = dev_df['conversation_id'].unique()

for conv_id in unique_dev_ids:
    conv_data = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')
    for current_turn in range(6, 11):
        target_row = conv_data[conv_data['turn_id'] == current_turn]
        if not target_row.empty:
            history_text = " [SEP] ".join(conv_data[conv_data['turn_id'] < current_turn]['text'].tolist())
            gen_text = get_corpus_response(history_text, target_row.iloc[0]['Emotion'],
                                           target_row.iloc[0]['Empathy'],
                                           target_row.iloc[0]['EmotionalPolarity'],
                                           train_df, train_embeddings)
            dev_predictions.append(gen_text)
            dev_references.append(target_row.iloc[0]['text'])

r_results = rouge.compute(predictions=dev_predictions, references=dev_references)
b_results = bleu.compute(predictions=dev_predictions, references=dev_references)
bert_results = bertscore.compute(predictions=dev_predictions, references=dev_references, lang="en")


print(f"ROUGE-L: {r_results['rougeL']:.4f}")
print(f"BLEU: {b_results['bleu']:.4f}")
print(f"BertScore (F1 Mean): {np.mean(bert_results['f1']):.4f}")

Evaluating on Validation (Dev) Set for metrics...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ROUGE-L: 0.1094
BLEU: 0.0101
BertScore (F1 Mean): 0.8600
